**Cell 1 -- Library Installation**

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate huggingface_hub peft

# Step 3: Code/DSL libraries
!pip install -q sqlglot bashlex sql-metadata tqdm nltk pyyaml

# Step 4: Evaluation libraries
!pip install -q code-bert-score selenium webdriver-manager

# Step 5: Ansible
!pip install -q ansible-lint

# Step 6: RL training (GRPO — used for training)
!pip install -q trl

# Step 7: Pin sympy LAST after everything else installed
!pip install -q sympy==1.13.3


# Step 8: Verify
print("Verifying critical imports...")
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationMixin
print("✅ transformers OK")
import sympy
print(f"✅ sympy version: {sympy.__version__}")
import torch
print(f"✅ torch version: {torch.__version__}")
print("\nAll libraries installed and verified.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━

**Cell 2 -- Google Drive Setup + Project Structure**

In [ ]:
from google.colab import auth, drive
import os, json, re, gc, random, copy
import torch

auth.authenticate_user()
try:
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted.")
except Exception as e:
    print(f"Drive note: {e}")

PROJECT_ROOT = "/content/drive/MyDrive/SLMFix_Project"
for d in ["generated_data", "checkpoints", "ansible_dataset", "spider_data", "bash_datasets"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

# ── File-naming helpers ─────────────────────────────────────────────────────
def final_path(subdir, filename):
    """Return absolute path with final_ prefix.
    ALL persistent outputs must use this so re-running never overwrites a
    completed file.  Idempotency rule: if the final_ file exists, skip.
    """
    return os.path.join(PROJECT_ROOT, subdir, f"final_{filename}")

def safe_load_json(path):
    """Load JSON if it exists, else return None."""
    if os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)
    return None

def safe_save_json(path, obj):
    """Save JSON and print confirmation."""
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2)
    print(f"Saved → {path}")


Mounted at /content/drive
Drive mounted.
Project root: /content/drive/MyDrive/SLMFix_Project


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/SLMFix_Project"))
print(os.listdir("/content/drive/MyDrive/SLMFix_Project/spider_data"))

['qwen_coder_7b_4bit', 'bash_datasets', 'spider_data', 'generated_data', 'checkpoints', 'ansible_dataset', 'qwen_coder_0.5b']
['.DS_Store', 'test_tables.json', 'dev_gold.sql', 'test_gold.sql', 'dev.json', 'README.txt', 'test.json', 'tables.json', 'spider_data', 'test_database']


**Cell 3 — Session keep-alive**

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function clickConnect() {
    document.querySelector("#top-toolbar > colab-connect-button")
        ?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(clickConnect, 60000);
'''))
print("Keep-alive active.")

<IPython.core.display.Javascript object>

Keep-alive active.


Cell 4 — Load datasets

SQL=Spider, Bash=NL2Bash, Ansible=constructed dataset.Train/eval strictly separated. **bold text**

In [ ]:
# SQL — Spider dataset
with open(os.path.join(PROJECT_ROOT, "spider_data/spider_data/train_spider.json")) as f:
    train_sdata = json.load(f)
with open(os.path.join(PROJECT_ROOT, "spider_data/dev.json")) as f:
    dev_sdata = json.load(f)   # EVAL ONLY

# Bash — NL2Bash dataset
with open(os.path.join(PROJECT_ROOT, "bash_datasets/train.json")) as f:
    train_bdata = json.load(f)
with open(os.path.join(PROJECT_ROOT, "bash_datasets/dev.json")) as f:
    dev_bdata = json.load(f)   # EVAL ONLY


print(f"SQL   — Train: {len(train_sdata)} | Dev: {len(dev_sdata)}")
print(f"Bash  — Train: {len(train_bdata)} | Dev: {len(dev_bdata)}")
print("NOTE: dev sets are held out for evaluation only — never used to generate training triples.")

SQL   — Train: 7000 | Dev: 1034
Bash  — Train: 8090 | Dev: 609
NOTE: dev sets are held out for evaluation only — never used to generate training triples.


In [ ]:
import os

# Path to the directory where train_spider.json is expected
expected_dir = os.path.join(PROJECT_ROOT, "spider_data/spider_data/")

print(f"Contents of {expected_dir}:")
if os.path.exists(expected_dir):
    for item in os.listdir(expected_dir):
        print(item)
else:
    print(f"Directory does not exist: {expected_dir}")


Contents of /content/drive/MyDrive/SLMFix_Project/spider_data/spider_data/:
train_spider.json
train_others.json
train_gold.sql
database


**Cell 5 — Schema helper (Spider/SQL)**

In [ ]:
import sqlite3

def get_db_schema(db_id):
    """Extract schema string from SQLite database (paper §5.1)."""
    db_path = os.path.join(PROJECT_ROOT, f"spider_data/spider_data/database/{db_id}/{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [t[0] for t in cursor.fetchall()]
        schema_text = ""
        for table in tables:
            cursor.execute(f"PRAGMA table_info({table});")
            columns = [col[1] for col in cursor.fetchall()]
            schema_text += f"Table: {table}, columns: {', '.join(columns)} | "
        conn.close()
        return schema_text
    except Exception as e:
        return f"Error reading schema: {e}"

def get_schema_dict(db_id):
    """Return schema as dict {table: [columns]} for sql-metadata validation."""
    db_path = os.path.join(PROJECT_ROOT, f"spider_data/spider_data/database/{db_id}/{db_id}.sqlite")
    schema = {}
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        for (table,) in cursor.fetchall():
            cursor.execute(f"PRAGMA table_info({table});")
            schema[table.lower()] = [col[1].lower() for col in cursor.fetchall()]
        conn.close()
    except Exception:
        pass
    return schema

print("Schema helpers ready.")

Schema helpers ready.


**Cell 6 — Output cleaning helpers**

In [ ]:
def clean_sql(raw):
    s = re.sub(r'```sql|```', '', raw)
    return s.strip().rstrip(';').strip()

def clean_bash(raw):
    s = re.sub(r'```bash|```', '', raw)
    return s.strip()

print("Cleaners ready.")

Cleaners ready.


**Cell 7 — SQL Validator**

In [ ]:
import sqlglot
from sqlglot import exp
from sqlglot.errors import TokenError as SqlglotTokenError

try:
    from sql_metadata import Parser as SqlMetaParser
    SQL_METADATA_AVAILABLE = True
except ImportError:
    SQL_METADATA_AVAILABLE = False
    print("sql-metadata not available — using SQLGlot fallback")

def validate_sql_combined(predicted_sql, db_id, schema_dict=None):
    """SQL validator — paper §3.2.3. Returns rl_reward (for GRPO) and fully_valid (for eval)."""
    sql = clean_sql(predicted_sql)
    result = {
        "static_valid": False, "static_error": None,
        "schema_valid": False, "schema_error": None,
        "runtime_valid": False, "runtime_error": None, "runtime_output": None,
        "rl_reward": 0.0, "fully_valid": False,
    }
    # Stage 1: SQLGlot syntax
    try:
        sqlglot.parse_one(sql, dialect="sqlite")
        result["static_valid"] = True
    except (sqlglot.errors.ParseError, SqlglotTokenError) as e:
        result["static_error"] = str(e)
        return result
    except Exception as e:
        result["static_error"] = str(e)
        return result

    # Stage 2: schema validation via sql-metadata (paper §3.2.3)
    if schema_dict is None and db_id:
        schema_dict = get_schema_dict(db_id)
    if schema_dict:
        all_tables = set(schema_dict.keys())
        all_cols   = {c for cols in schema_dict.values() for c in cols}
        ok = True
        if SQL_METADATA_AVAILABLE:
            try:
                p = SqlMetaParser(sql)
                bad_t = {t.lower() for t in p.tables} - all_tables
                bad_c = {c.lower().split('.')[-1] for c in p.columns} - all_cols - {'*'}
                if bad_t:
                    result["schema_error"] = f"Unknown tables: {bad_t}"; ok = False
                elif bad_c:
                    result["schema_error"] = f"Unknown columns: {bad_c}"; ok = False
            except Exception:
                pass  # sql-metadata parse failure — fall through to SQLGlot fallback
        if ok:
            try:
                parsed = sqlglot.parse_one(sql, dialect="sqlite")
                bad_t = {t.name.lower() for t in parsed.find_all(exp.Table)} - all_tables
                if bad_t:
                    result["schema_error"] = f"Unknown tables (fb): {bad_t}"; ok = False
            except Exception:
                pass
        if not ok:
            return result

    result["schema_valid"] = True
    result["rl_reward"]    = 1.0

    # Stage 3: execution — EVALUATION ONLY, never used for RL reward
    db_path = os.path.join(PROJECT_ROOT, f"spider_data/spider_data/database/{db_id}/{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        rows = conn.execute(sql).fetchall()
        conn.close()
        result["runtime_valid"] = True
        result["runtime_output"] = rows
        result["fully_valid"]   = True
    except Exception as e:
        result["runtime_error"] = str(e)
    return result

# Quick smoke test
r = validate_sql_combined("SELECT count(*) FROM head WHERE age > 56", "department_management")
print(f"Static: {r['static_valid']} | Schema: {r['schema_valid']} | rl_reward: {r['rl_reward']} | exec: {r['runtime_valid']}")

Static: True | Schema: True | rl_reward: 1.0 | exec: True


**Cell 8 — Bash Validator**

In [ ]:
import bashlex

COMMON_UTILS = {
    "find","grep","sed","awk","ls","cat","echo","cp","mv","rm","mkdir","chmod","chown",
    "tar","gzip","gunzip","sort","uniq","wc","head","tail","cut","tr","xargs","tee",
    "diff","touch","pwd","cd","export","read","printf","test","curl","wget","ssh",
    "scp","rsync","zip","unzip","du","df","ps","kill","which","file","stat","date",
    "basename","dirname","env","source","alias","type","uname","hostname","id","groups"
}

def validate_bash_static(command):
    """Static Bash validator — paper §3.2.2. Never executes."""
    cmd = clean_bash(command)
    result = {"static_valid": False, "static_error": None, "rl_reward": 0.0}
    if not cmd.strip():
        result["static_error"] = "Empty command"; return result
    if '<<' in cmd:
        result["static_error"] = "here-doc excluded"; return result
    if '<(' in cmd or '>(' in cmd:
        result["static_error"] = "process substitution excluded"; return result
    try:
        bashlex.parse(cmd)
        result["static_valid"] = True
        result["rl_reward"]    = 1.0
    except Exception as e:
        result["static_error"] = str(e)
    return result

print("Bash validator ready.")

Bash validator ready.


**Cell 9 — AST Similarity Metrics**

In [ ]:
# ── SQL AST similarity — paper §3.3.3 ─────────────────────────────────────
from sqlglot.diff import Keep as DiffKeep
def compute_sql_ast_similarity(predicted_sql, ground_truth_sql):
    """
    SQL AST similarity — paper §3.3.3.
    score = 1/(1+edits).
    Column aliases removed, table aliases replaced with original names.
    """
    try:
        def resolve_aliases(ast):
            # Step A — build table alias map {alias: original_table_name}
            alias_map = {}
            for node in ast.walk():
                if isinstance(node, sqlglot.exp.Table):
                    if node.alias:
                        alias_map[node.alias] = node.name

            def replace_table_aliases(node):
                # Step B — replace table alias references in Column nodes
                if isinstance(node, sqlglot.exp.Column):
                    if node.table and node.table in alias_map:
                        return sqlglot.exp.Column(
                            this=node.this,
                            table=sqlglot.exp.Identifier(
                                this=alias_map[node.table],
                                quoted=False
                            )
                        )
                # Step C — remove column aliases e.g. age AS person_age
                if isinstance(node, sqlglot.exp.Alias):
                    if not isinstance(node.parent, sqlglot.exp.Table):
                        return node.this
                # Step D — remove table alias from FROM clause
                if isinstance(node, sqlglot.exp.Table):
                    return sqlglot.exp.Table(this=node.this)
                return node

            return ast.transform(replace_table_aliases)

        pred_ast = sqlglot.parse_one(clean_sql(predicted_sql), dialect="sqlite")
        gold_ast = sqlglot.parse_one(clean_sql(ground_truth_sql), dialect="sqlite")
        pred_ast = resolve_aliases(pred_ast)
        gold_ast = resolve_aliases(gold_ast)

        actual_edits = [
            d for d in sqlglot.diff(pred_ast, gold_ast)
            if not isinstance(d, DiffKeep)
        ]
        return 1.0 / (1.0 + len(actual_edits))

    except Exception:
        return 0.0


# ── Bash AST similarity — paper §3.3.2 ────────────────────────────────────
def _bash_to_kv(cmd_str):
    """Parse single bash command into {option: argument} dict."""
    try:
        parts = bashlex.parse(clean_bash(cmd_str))
    except Exception:
        return {}
    kv = {}
    for part in parts:
        if not hasattr(part, 'parts'):
            continue
        words = [n.word for n in part.parts if hasattr(n, 'word')]
        i = 0
        while i < len(words):
            w = words[i]
            if w.startswith('-') and i + 1 < len(words):
                kv[w] = words[i+1]
                i += 2
            else:
                kv[w] = w
                i += 1
    return kv

def compute_bash_ast_similarity(predicted_bash, gold_bash):
    """Paper §3.3.2: fraction of gold key-value pairs present in prediction."""
    pred_kv = _bash_to_kv(predicted_bash)
    gold_kv = _bash_to_kv(gold_bash)
    if not gold_kv:
        # Fallback: token F1
        pred_t = set(clean_bash(predicted_bash).split())
        gold_t = set(clean_bash(gold_bash).split())
        if not gold_t:
            return 0.0
        ov = pred_t & gold_t
        p = len(ov) / len(pred_t) if pred_t else 0
        r = len(ov) / len(gold_t)
        return 2*p*r/(p+r) if (p+r) > 0 else 0.0
    matches = sum(1 for k,v in gold_kv.items() if pred_kv.get(k) == v)
    return matches / len(gold_kv)

# Smoke tests
print("SQL AST similarity:")
print(f"  Identical : {compute_sql_ast_similarity('SELECT count(*) FROM head', 'SELECT count(*) FROM head'):.4f}  (expect 1.0)")
print(f"  Different : {compute_sql_ast_similarity('SELECT name FROM head', 'SELECT count(*) FROM department'):.4f}  (expect <1.0)")
print()
print("Bash AST similarity:")
print(f"  Identical : {compute_bash_ast_similarity('ls -la /tmp', 'ls -la /tmp'):.4f}  (expect 1.0)")

SQL AST similarity:
  Identical : 1.0000  (expect 1.0)
  Different : 0.1000  (expect <1.0)

Bash AST similarity:
  Identical : 1.0000  (expect 1.0)


**Cell 10 — Adaptive Reward Functions**

In [ ]:
def compute_reward(static_score, semantic_score, batch_pass_rate):
    """Adaptive reward — paper §3.1. Works for all three languages."""
    pr = batch_pass_rate
    return (1.0 - pr) * static_score + pr * semantic_score

def compute_sql_reward(predicted_sql, ground_truth_sql, db_id, batch_pass_rate):
    return compute_reward(
        validate_sql_combined(predicted_sql, db_id)["rl_reward"],
        compute_sql_ast_similarity(predicted_sql, ground_truth_sql),
        batch_pass_rate
    )

def compute_bash_reward(predicted_bash, ground_truth_bash, batch_pass_rate):
    return compute_reward(
        validate_bash_static(predicted_bash)["rl_reward"],
        compute_bash_ast_similarity(predicted_bash, ground_truth_bash),
        batch_pass_rate
    )


print("Reward functions ready.")
print("Formula: r = (1-pr)*static + pr*semantic  (pr = batch pass rate)")

Reward functions ready.
Formula: r = (1-pr)*static + pr*semantic  (pr = batch pass rate)


**Cell 12 — Download Qwen-2.5-Coder-0.5B**

In [ ]:
#0.5B qwen Download


from huggingface_hub import snapshot_download

SLM_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_0.5b")
os.makedirs(SLM_DIR, exist_ok=True)

print("Downloading Qwen2.5-Coder-0.5B-Instruct from HuggingFace...")
print("This will take a few minutes...")

snapshot_download(
    repo_id="Qwen/Qwen2.5-Coder-0.5B-Instruct",
    local_dir=SLM_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],  # skip non-pytorch files
)

print(f"Download complete!")
print(f"Files saved to: {SLM_DIR}")
print(f"Files: {os.listdir(SLM_DIR)}")

This will take a few minutes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Download complete!
Files saved to: /content/drive/MyDrive/SLMFix_Project/qwen_coder_0.5b
Files: ['.cache', 'README.md', '.gitattributes', 'config.json', 'generation_config.json', 'LICENSE', 'merges.txt', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'model.safetensors']


**Cell 13: Verify Model Paths**

In [ ]:
import os

SLM_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_0.5b")
print("Path:", SLM_DIR)
print("Exists:", os.path.exists(SLM_DIR))

if os.path.exists(SLM_DIR):
    print("Files:", os.listdir(SLM_DIR))
else:
    print("Directory not found — need to download model first")


Path: /content/drive/MyDrive/SLMFix_Project/qwen_coder_0.5b
Exists: True
Files: ['.cache', 'README.md', '.gitattributes', 'config.json', 'generation_config.json', 'LICENSE', 'merges.txt', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'model.safetensors']


**Cell 14 — Load 0.5B Fixer Model**

In [ ]:
MODEL_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_7b_4bit")
print("7B exists:", os.path.exists(MODEL_DIR))
if os.path.exists(MODEL_DIR):
    print("Files:", os.listdir(MODEL_DIR))

7B exists: True
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


**Cell 15 -- Load 0.5B Fixer**

In [ ]:
#!pip install -q "transformers==4.47.0" "huggingface_hub==0.26.5" "tokenizers==0.21.0"
#import importlib, transformers, huggingface_hub, tokenizers
#importlib.reload(transformers)
#print("transformers:", transformers.__version__)
#print("huggingface_hub:", huggingface_hub.__version__)
#print("tokenizers:", tokenizers.__version__)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

SLM_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_0.5b")  # your local path

print("Loading Qwen-2.5-Coder-0.5B (trainable fixer SLM)...")
fixer_tokenizer = AutoTokenizer.from_pretrained(SLM_DIR, local_files_only=True)
if fixer_tokenizer.pad_token_id is None:
    fixer_tokenizer.pad_token_id = fixer_tokenizer.eos_token_id
if fixer_tokenizer.pad_token_id is None:
    fixer_tokenizer.pad_token_id = 0
model = AutoModelForCausalLM.from_pretrained(
    SLM_DIR,
    device_map="auto",
    local_files_only=True,
    low_cpu_mem_usage=True,
)
model.train()
print("0.5B fixer SLM ready. This model WILL be trained.")


Loading Qwen-2.5-Coder-0.5B (trainable fixer SLM)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

0.5B fixer SLM ready. This model WILL be trained.


**Cell 16 -- Load 7B Proposer**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_7b_4bit")

print("Loading Qwen-2.5-Coder-7B-Instruct (4-bit quantized, FROZEN)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
proposer_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map="auto",
    local_files_only=True,
    low_cpu_mem_usage=True,
)
print(f"7B proposer ready. This model is NEVER trained.")

# ── Generation functions — paper Appendix B.1 ────────────────────────────────
def _generate(tok, mdl, messages, max_new_tokens=150, temperature=0.0):
    """Core generation helper. temperature=0 => greedy (paper §5.2 inference)."""
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", max_length=2048, truncation=True).to(mdl.device)
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0), temperature=temperature if temperature > 0 else None,
            pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id
        )
    return tok.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def generate_sql(question, schema, tok=None, mdl=None, temperature=0.0):
    """SQL generation — paper Appendix B.1."""
    return _generate(tok or tokenizer, mdl or proposer_model, [
        {"role": "system", "content": "You are an expert in SQL. The user will give you a task description and ask you to generate a SQL command to complete the given task. You only need to output the content of the command."},
        {"role": "user", "content": f"Database Schema:\n{schema}\n\nTask: {question}\n\nAnswer: ```sql"}
    ], max_new_tokens=150, temperature=temperature)

def generate_bash(nl_desc, tok=None, mdl=None, temperature=0.0):
    """Bash generation — paper Appendix B.1."""
    return _generate(tok or tokenizer, mdl or proposer_model, [
        {"role": "system", "content": "You are an expert in Bash. The user will give you a task description and ask you to generate a bash command to complete the given task. You only need to output the content of the command."},
        {"role": "user", "content": f"Task: {nl_desc}\n\nAnswer: ```bash"}
    ], max_new_tokens=100, temperature=temperature)


# Quick test
if train_sdata:
    s = train_sdata[0]
    res = generate_sql(s["question"], get_db_schema(s["db_id"]))
    print(f"Test SQL generation: {res[:80]}")

Loading Qwen-2.5-Coder-7B-Instruct (4-bit quantized, FROZEN)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

7B proposer ready. This model is NEVER trained.
Test SQL generation: ```sql
SELECT COUNT(*) FROM head WHERE age > 56;
```


**Cell 17 -- Device Check**

In [ ]:
#test
print("proposer_model device:", next(proposer_model.parameters()).device)
print("fixer model device:", next(model.parameters()).device)
print("tokenizer:", tokenizer)
print("fixer_tokenizer:", fixer_tokenizer)

proposer_model device: cuda:0
fixer model device: cuda:0
tokenizer: Qwen2Tokenizer(name_or_path='/content/drive/MyDrive/SLMFix_Project/qwen_coder_7b_4bit', vocab_size=151643, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False,

# GRPO training part

**Cell 18 -- Install Shell Check**

In [ ]:
!apt-get install -y shellcheck -q
!shellcheck --version

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  shellcheck
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 2,359 kB of archives.
After this operation, 16.3 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 shellcheck amd64 0.8.0-2 [2,359 kB]
Fetched 2,359 kB in 1s (3,425 kB/s)
Selecting previously unselected package shellcheck.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../shellcheck_0.8.0-2_amd64.deb ...
Unpacking shellcheck (0.8.0-2) ...
Setting up shellcheck (0.8.0-2) ...
Processing triggers for man-db (2.10.2-1) ...
ShellCheck - shell script analysis tool
version: 0.8.0
license: GNU General Public License, version 3
website: https://www.shellcheck.net


**Cell 19 -- Shellcheck Validator**

In [ ]:
import subprocess
import json

def validate_bash_heuristic(command):
    """
    ShellCheck-based validator — paper §3.2.2.
    Only flags SC1xxx errors (syntax/parse errors that cause failure).
    Ignores style/lint warnings (SC2xxx, SC3xxx).
    Used for triple construction only, never for RL reward.
    """
    cmd = clean_bash(command).strip()
    if not cmd:
        return False, "Empty command"

    try:
        result = subprocess.run(
            ["shellcheck", "-f", "json", "-s", "bash", "-"],
            input=cmd,
            capture_output=True,
            text=True,
            timeout=10
        )

        if result.returncode == 0:
            return True, None

        issues = json.loads(result.stdout) if result.stdout else []

        # Only care about SC1xxx — actual syntax errors
        fatal_errors = [
            i for i in issues
            if 1000 <= i.get("code", 0) < 2000
        ]

        if fatal_errors:
            msg = fatal_errors[0].get("message", "ShellCheck syntax error")
            return False, f"SC{fatal_errors[0]['code']}: {msg}"

        return True, None  # only style warnings, command is fine

    except FileNotFoundError:
        return True, None  # shellcheck not found, skip
    except Exception as e:
        return False, str(e)

# Quick test
print(validate_bash_heuristic("find . -name '*.txt'"))   # should be True
print(validate_bash_heuristic(""))                        # should be False
print(validate_bash_heuristic("ls -la /tmp"))             # should be True

(True, None)
(False, 'Empty command')
(True, None)


**Cell 20 -- Load pre-saved Triples**

In [ ]:
import json

with open(os.path.join(PROJECT_ROOT, "generated_data/training_triples.json")) as f:
    sql_triples = json.load(f)

with open(os.path.join(PROJECT_ROOT, "generated_data/bash_training_triples.json")) as f:
    bash_triples = json.load(f)

print("=== SQL Triple Sample ===")
print(json.dumps(sql_triples[0], indent=2))

print("\n=== Bash Triple Sample ===")
print(json.dumps(bash_triples[0], indent=2))

=== SQL Triple Sample ===
{
  "question": "List the name, born state and age of the heads of departments ordered by age.",
  "db_id": "department_management",
  "schema": "Table: department, columns: Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees | Table: head, columns: head_ID, name, born_state, age | Table: management, columns: department_ID, head_ID, temporary_acting | ",
  "broken_sql": "```sql\nSELECT h.name, h.born_state, h.age \nFROM head h \nJOIN management m ON h.head_id = m.head_id \nORDER BY h.age;\n```",
  "error_msg": "Low semantic similarity: 0.700",
  "failure_type": "semantic_mismatch",
  "gold_sql": "SELECT name ,  born_state ,  age FROM head ORDER BY age",
  "rl_reward": 0.0
}

=== Bash Triple Sample ===
{
  "invocation": "Do a dry run of renaming file extension '.andnav' to '.tile' for all files/directories under current directory tree",
  "broken_cmd": "```bash\nfind . -type f -name \"*.andnav\" -exec echo mv {} {}.tile \\;\n```",
  "error

**Cell 21 -- Bash Similarity Helper Func**

In [ ]:
# ⚠️ KEEP
#-----------------------
def compute_bash_similarity(predicted_bash, gold_bash):
    """
    Token-level F1 between generated and gold command.
    Used as the semantic score in the PPO reward function.
    """
    pred_tokens = set(predicted_bash.strip().split())
    gold_tokens = set(gold_bash.strip().split())

    if not pred_tokens or not gold_tokens:
        return 0.0

    overlap   = pred_tokens & gold_tokens
    precision = len(overlap) / len(pred_tokens)
    recall    = len(overlap) / len(gold_tokens)

    if precision + recall == 0:
        return 0.0

    f1 = 2 * precision * recall / (precision + recall)
    return f1

**Cell 22 -- Validate Bash Combined Helper Func**

In [ ]:
# ⚠️ KEEP
#-----------------------
def validate_bash_combined(predicted_bash):
    """
    Stage 1 — bashlex static parse  → sets rl_reward (paper §3.2.2)
    Stage 2 — ShellCheck heuristic  → used for triple construction only
    """
    command = clean_bash(predicted_bash)

    result = {
        "static_valid":    False,
        "static_error":    None,
        "heuristic_valid": False,
        "heuristic_error": None,
        "rl_reward":       0.0,
        "fully_valid":     False,
    }

    # Stage 1: bashlex static parse — paper §3.2.2
    static_result = validate_bash_static(command)   # ← was _bash_valid_dict, now fixed
    static_ok  = static_result["static_valid"]
    static_err = static_result["static_error"]

    if not static_ok:
        result["static_error"] = static_err
        return result

    result["static_valid"] = True
    result["rl_reward"]    = 1.0

    # Stage 2: ShellCheck — triple construction only, never for RL reward
    heuristic_ok, heuristic_err = validate_bash_heuristic(command)
    result["heuristic_valid"] = heuristic_ok
    result["heuristic_error"] = heuristic_err
    result["fully_valid"]     = heuristic_ok

    return result

**Cell 23 -- Compute Bash reward Helper Func**

In [ ]:
# Not needed ⚠️ KEEP
#-----------------------
def compute_bash_reward(predicted_bash, gold_bash, batch_pass_rate):
    """
    PPO reward = (1-pr) × static_score + pr × semantic_score
    Adaptive weighting: early training focuses on syntax, later on semantics.
    """
    static_score   = validate_bash_combined(predicted_bash)["rl_reward"]
    semantic_score = compute_bash_similarity(predicted_bash, gold_bash)
    pr             = batch_pass_rate
    return (1 - pr) * static_score + pr * semantic_score


# GRPO Training Cell

**Cell 24 -- Free the 7B model from VRAM**

In [ ]:
# Free the 7B proposer — not needed during PPO
import gc
del proposer_model
gc.collect()
torch.cuda.empty_cache()
print("VRAM freed:", torch.cuda.memory_allocated()/1e9, "GB used now")

VRAM freed: 0.997665792 GB used now


**Cell 25 -- The GRPO Training Loop (30 epochs)**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  GRPO TRAINING LOOP (PAPER-ALIGNED + T4 SAFE + CHECKPOINTING)            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, gc, json, random, torch
import torch.nn.functional as F
from tqdm import tqdm


# HYPERPARAMS
# ───────────────────────────────────────────────────────────────────────────
EPOCHS        = 15
GROUP_SIZE    = 2
ACCUM_STEPS   = 8
KL_COEF       = 0.02
MAX_NEW_TOKENS = 40
MAX_PROMPT_LEN = 384
MAX_SAMPLES   = 120   # per epoch → ~3-4 hrs total


# ───────────────────────────────────────────────────────────────────────────
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint dir:", CKPT_DIR)

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()
gc.collect()

# ── Use model already loaded in Cell 15 ────────────────────────────────────
actor = model        # trainable — loaded in Cell 15
actor.gradient_checkpointing_enable()
actor.train()

# ── Frozen reference on CPU ─────────────────────────────────────────────────
from transformers import AutoModelForCausalLM
ref = AutoModelForCausalLM.from_pretrained(
    SLM_DIR,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)
ref.eval()

optimizer = torch.optim.AdamW(actor.parameters(), lr=2e-6)

# ── RESUME FROM CHECKPOINT ──────────────────────────────────────────────────
RESUME_FROM_EPOCH = 0

for e in range(EPOCHS, 0, -1):
    p = os.path.join(CKPT_DIR, f"epoch_{e}")
    if os.path.exists(p):
        RESUME_FROM_EPOCH = e
        print(f"⚡ Found checkpoint at epoch {e} — resuming...")

        actor.load_state_dict(
            AutoModelForCausalLM.from_pretrained(
                p, torch_dtype=torch.float16, trust_remote_code=True
            ).state_dict()
        )

        opt_state = torch.load(os.path.join(p, "optimizer.pt"))
        optimizer.load_state_dict(opt_state["optimizer"])

        print(f"✅ Resumed from epoch {RESUME_FROM_EPOCH}")
        break

if RESUME_FROM_EPOCH == 0:
    print("🆕 No checkpoint found — starting fresh")


# ───────────────────────────────────────────────────────────────────────────
# DATA
# ───────────────────────────────────────────────────────────────────────────
with open(os.path.join(PROJECT_ROOT, "generated_data/training_triples.json")) as f:
    sql_triples = json.load(f)

with open(os.path.join(PROJECT_ROOT, "generated_data/bash_training_triples.json")) as f:
    bash_triples = json.load(f)

data = []
for t in sql_triples:
    t["language"] = "sql"
    data.append(t)
for t in bash_triples:
    t["language"] = "bash"
    data.append(t)

random.shuffle(data)
print(f"Total triples: {len(data)} | SQL: {len(sql_triples)} | Bash: {len(bash_triples)}")

# ───────────────────────────────────────────────────────────────────────────


# ───────────────────────────────────────────────────────────────────────────
def save_checkpoint(epoch):
    path = os.path.join(CKPT_DIR, f"epoch_{epoch}")
    os.makedirs(path, exist_ok=True)
    print(f"💾 Saving checkpoint → {path}")
    actor.save_pretrained(path)
    fixer_tokenizer.save_pretrained(path)
    torch.save({
        "optimizer": optimizer.state_dict(),
        "epoch": epoch
    }, os.path.join(path, "optimizer.pt"))
    print("✅ Saved")

# ───────────────────────────────────────────────────────────────────────────
def build_prompt(t):
    if t["language"] == "sql":
        msgs = [
            {"role": "system", "content": "Fix SQL. Output only SQL."},
            {"role": "user", "content":
                f"{t['question']}\n{t['schema']}\n{t['broken_sql']}\n{t['error_msg']}\n```sql"}
        ]
    else:
        msgs = [
            {"role": "system", "content": "Fix Bash. Output only command."},
            {"role": "user", "content":
                f"{t['invocation']}\n{t['broken_cmd']}\n{t['error_msg']}\n```bash"}
        ]
    return fixer_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# ───────────────────────────────────────────────────────────────────────────
def logprob(mdl, input_ids):
    logits = mdl(input_ids).logits[:, :-1, :]
    return F.log_softmax(logits, dim=-1)

# ───────────────────────────────────────────────────────────────────────────
def sample(prompt):
    enc = fixer_tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_PROMPT_LEN
    ).to("cuda")

    prompt_len = enc["input_ids"].shape[1]
    input_ids  = enc["input_ids"].repeat(GROUP_SIZE, 1)
    attn       = enc["attention_mask"].repeat(GROUP_SIZE, 1)

    actor.eval()
    with torch.no_grad():
        out = actor.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=fixer_tokenizer.pad_token_id,
            use_cache=False
        )
    actor.train()

    gens, texts = [], []
    for i in range(GROUP_SIZE):
        g = out[i:i+1, prompt_len:]
        if g.shape[1] == 0:
            continue
        texts.append(fixer_tokenizer.decode(g[0], skip_special_tokens=True))
        gens.append((enc["input_ids"].cpu(), g.cpu()))

    return texts, gens

# ───────────────────────────────────────────────────────────────────────────
def grpo_loss(gen_list, rewards):
    if len(gen_list) == 0:
        return None, 0.0

    device  = "cuda"
    p_ids   = gen_list[0][0].to(device)
    actor_scores, ref_scores = [], []

    for (_, g), _ in zip(gen_list, rewards):
        g    = g.to(device)
        full = torch.cat([p_ids, g], dim=1)

        # Actor logprob
        lp_actor = logprob(actor, full)
        lp_actor = lp_actor[:, -g.shape[1]:, :]
        g_safe   = g.clamp(0, lp_actor.size(-1) - 1).long()
        actor_lp = lp_actor.gather(2, g_safe.unsqueeze(-1)).mean()

        # Reference logprob (CPU)
        with torch.no_grad():
            lp_ref = logprob(ref, full.cpu())
            lp_ref = lp_ref[:, -g.shape[1]:, :]
            ref_lp = lp_ref.gather(2, g_safe.cpu().unsqueeze(-1)).mean()

        actor_scores.append(actor_lp)
        ref_scores.append(ref_lp)

    actor_scores = torch.stack(actor_scores)
    ref_scores   = torch.stack(ref_scores).to(device)

    # GRPO advantage — paper §3.1
    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)
    adv       = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)

    policy_loss = -(actor_scores * adv).mean()
    kl          = (actor_scores - ref_scores).mean()
    loss        = policy_loss + KL_COEF * kl

    return loss, kl.item()

# ───────────────────────────────────────────────────────────────────────────
print("🚀 GRPO TRAINING STARTED")

for epoch in range(RESUME_FROM_EPOCH, EPOCHS):
    random.shuffle(data)
    optimizer.zero_grad()
    step = 0
    epoch_rewards = []

    for i in tqdm(range(min(MAX_SAMPLES, len(data)))):
        t      = data[i]
        prompt = build_prompt(t)

        try:
            responses, gen_list = sample(prompt)
        except Exception as e:
            continue

        if len(responses) < 2:
            continue


        # ── REAL REWARD — paper §3.1 ──
        # Step 1: pass rate for this group
        pass_count = 0
        for r in responses:
            if t["language"] == "sql":
                if validate_sql_combined(r, t["db_id"])["static_valid"]:
                    pass_count += 1
            else:
                if validate_bash_static(r)["static_valid"]:
                    pass_count += 1

        batch_pass_rate = pass_count / len(responses)

        # Step 2: adaptive reward — r = (1-pr)*static + pr*semantic
        rewards = []
        for r in responses:
            if t["language"] == "sql":
                rew = compute_sql_reward(r, t["gold_sql"], t["db_id"], batch_pass_rate)
            else:
                rew = compute_bash_reward(r, t["gold_cmd"], batch_pass_rate)
            rewards.append(rew)


        loss, kl = grpo_loss(gen_list, rewards)
        if loss is None:
            continue

        (loss / ACCUM_STEPS).backward()
        epoch_rewards.extend(rewards)
        step += 1

        if step % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            torch.cuda.empty_cache()

    avg_reward = sum(epoch_rewards) / len(epoch_rewards) if epoch_rewards else 0
    print(f"Epoch {epoch+1}/{EPOCHS} | AvgReward: {avg_reward:.4f} | KL: {kl:.4f}")

    if (epoch + 1) % 3 == 0:        #save checkpoint per every 3 epochs
        save_checkpoint(epoch + 1)

# Final save
save_checkpoint(EPOCHS)
print("✅ GRPO TRAINING COMPLETE")

**Cell 26 -- Reload 7B PROPOSER**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_7b_4bit")

print("Loading Qwen-2.5-Coder-7B-Instruct (4-bit quantized, FROZEN)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
proposer_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map="auto",
    local_files_only=True,
    low_cpu_mem_usage=True,
)
print("7B proposer ready.")

Loading Qwen-2.5-Coder-7B-Instruct (4-bit quantized, FROZEN)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

7B proposer ready.


**Cell 27 -- Load 15th epoch (Trained GRPO Fixer)**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

FIXER_PATH = os.path.join(PROJECT_ROOT, "checkpoints/epoch_15")

print("Loading trained GRPO fixer (epoch 15)...")
eval_fixer = AutoModelForCausalLM.from_pretrained(
    FIXER_PATH,
    local_files_only=True,
    torch_dtype=torch.float16
)
eval_fixer.eval()
print("GRPO fixer loaded from epoch 15.")

Loading trained GRPO fixer (epoch 15)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

GRPO fixer loaded from epoch 15.


# EVALUATION CELL

**Cell 28 -- Evaluation Helper Functions**

In [ ]:
import torch, gc
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from code_bert_score import score as cbs_score

# ── Constants from training — must match Cell 25 ──────────────────────────
MAX_PROMPT_LEN = 384
MAX_NEW_TOKENS = 40

In [ ]:
def build_prompt(t):
    """Same prompt format as Cell 25 training."""
    if t["language"] == "sql":
        msgs = [
            {"role": "system", "content": "Fix SQL. Output only SQL."},
            {"role": "user", "content":
                f"{t['question']}\n{t['schema']}\n{t['broken_sql']}\n{t['error_msg']}\n```sql"}
        ]
    else:
        msgs = [
            {"role": "system", "content": "Fix Bash. Output only command."},
            {"role": "user", "content":
                f"{t['invocation']}\n{t['broken_cmd']}\n{t['error_msg']}\n```bash"}
        ]
    return fixer_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def fixer_inference(triple):
    """Run trained GRPO fixer on CPU to avoid VRAM conflict."""
    prompt = build_prompt(triple)
    inputs = fixer_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_PROMPT_LEN,
        truncation=True
    )

    with torch.no_grad():
        outputs = eval_fixer.to("cpu").generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=fixer_tokenizer.eos_token_id,
        )

    generated_ids = outputs[:, inputs["input_ids"].shape[1]:]
    return fixer_tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()


def compute_bleu(predicted, gold):
    """Sentence-level BLEU — paper §5.1."""
    smoothie = SmoothingFunction().method1
    return sentence_bleu([gold.split()], predicted.split(), smoothing_function=smoothie)


def evaluate_slmfix(dev_data, language, max_samples=None, use_fixer=True):
    """
    Full evaluation — paper §5.3.
    Base mode:   7B proposer generates, no fixer.
    SLMFix mode: 7B proposer generates, GRPO fixer fixes if broken.
    """
    if max_samples:
        dev_data = dev_data[:max_samples]

    pass_count = 0
    fix_attempted = 0    # track how many samples the fixer actually ran on
    fix_recovered = 0    # track how many failed samples the fixer recovered
    bleu_scores, ast_scores = [], []
    predictions, gold_refs = [], []

    print(f"\nEvaluating {language.upper()} | "
          f"Samples: {len(dev_data)} | "
          f"Mode: {'SLMFix-GRPO' if use_fixer else 'Base'}")

    for sample in tqdm(dev_data):
        try:
            if language == "sql":
                db_id     = sample["db_id"]
                schema    = get_db_schema(db_id)
                gold      = sample["query"]
                base_pred = generate_sql(
                    sample["question"], schema,
                    tok=tokenizer, mdl=proposer_model
                )
            else:
                gold      = sample["bash"]
                base_pred = generate_bash(
                    sample["nl"],
                    tok=tokenizer, mdl=proposer_model
                )

            torch.cuda.empty_cache()
            gc.collect()

            # ── Apply fixer if needed ──────────────────────────────────
            if use_fixer:
                if language == "sql":
                    val     = validate_sql_combined(base_pred, db_id)
                    ast_sim = compute_sql_ast_similarity(base_pred, gold)

                    # LEVEL 1 FIX: trigger on validation failure OR low
                    # AST similarity — catches semantic failures the 7B
                    # model makes even when syntax is valid
                    needs_fix = not val["fully_valid"] or ast_sim < 0.7

                    error_msg = (
                        val.get("static_error")
                        or val.get("schema_error")
                        or val.get("runtime_error")
                        or f"Low semantic similarity: {ast_sim:.3f}"
                    )

                    if needs_fix:
                        fix_attempted += 1
                        triple = {
                            "language":   "sql",
                            "question":   sample["question"],
                            "schema":     schema,
                            "broken_sql": base_pred,
                            "error_msg":  error_msg,
                            "db_id":      db_id,
                        }
                        fixed_pred = clean_sql(fixer_inference(triple))

                        # LEVEL 1 FIX: only keep fix if it passes
                        # validation when base didn't, OR if it improves
                        # AST similarity — never make a passing output fail
                        fixed_val     = validate_sql_combined(fixed_pred, db_id)
                        fixed_ast_sim = compute_sql_ast_similarity(fixed_pred, gold)

                        base_passed = val["fully_valid"]
                        fix_passes  = fixed_val["fully_valid"]

                        if not base_passed and fix_passes:
                            # fixer recovered a failing sample — always accept
                            final_pred = fixed_pred
                            fix_recovered += 1
                        elif base_passed and fix_passes and fixed_ast_sim > ast_sim:
                            # base already passed but fixer improved AST — accept
                            final_pred = fixed_pred
                        else:
                            # fixer did not improve — keep base to avoid regression
                            final_pred = base_pred
                    else:
                        final_pred = base_pred

                else:
                    val = validate_bash_static(base_pred)
                    if isinstance(val, dict):
                        needs_fix = not val["static_valid"]
                        error_msg = val.get("static_error") or ""
                    else:
                        needs_fix = not val[0]
                        error_msg = val[1] or ""

                    if needs_fix:
                        fix_attempted += 1
                        triple = {
                            "language":   "bash",
                            "invocation": sample["nl"],
                            "broken_cmd": base_pred,
                            "error_msg":  error_msg,
                        }
                        fixed_pred = clean_bash(fixer_inference(triple))

                        # For Bash: accept fix if it passes static validation
                        fixed_v   = validate_bash_static(fixed_pred)
                        fix_ok    = fixed_v["static_valid"] if isinstance(fixed_v, dict) else fixed_v[0]

                        if fix_ok:
                            final_pred = fixed_pred
                            fix_recovered += 1
                        else:
                            final_pred = base_pred
                    else:
                        final_pred = base_pred
            else:
                final_pred = base_pred

            # ── Score final prediction ─────────────────────────────────
            if language == "sql":
                if validate_sql_combined(final_pred, db_id)["fully_valid"]:
                    pass_count += 1
                ast_scores.append(compute_sql_ast_similarity(final_pred, gold))
            else:
                v = validate_bash_static(final_pred)
                static_ok = v["static_valid"] if isinstance(v, dict) else v[0]
                if static_ok:
                    pass_count += 1
                ast_scores.append(compute_bash_ast_similarity(final_pred, gold))

            bleu_scores.append(compute_bleu(final_pred, gold))
            predictions.append(final_pred)
            gold_refs.append(gold)

        except Exception as e:
            print(f"  [skip] {e}")
            bleu_scores.append(0.0)
            ast_scores.append(0.0)
            predictions.append("")
            gold_refs.append(gold if 'gold' in dir() else "")
            continue

    print("Computing CodeBERTScore...")
    try:
        cbs_output = cbs_score(predictions, gold_refs, lang="python", verbose=False)
        if len(cbs_output) == 4:
            _, _, cbs, _ = cbs_output
        else:
            _, _, cbs = cbs_output
        cbs_mean = cbs.mean().item()
    except Exception as e:
        print(f"CodeBERTScore failed: {e}")
        cbs_mean = 0.0

    n = len(dev_data)
    results = {
        "pass_rate":       pass_count / n,
        "bleu":            sum(bleu_scores) / n,
        "code_bert_score": cbs_mean,
        "ast_diff":        sum(ast_scores) / n,

    }

    print(f"\nResults — {language.upper()} | {'SLMFix-GRPO' if use_fixer else 'Base'}")
    print(f"  Pass Rate:      {results['pass_rate']*100:.2f}%")
    print(f"  BLEU:           {results['bleu']:.4f}")
    print(f"  CodeBERTScore:  {results['code_bert_score']:.4f}")
    print(f"  AST Similarity: {results['ast_diff']:.4f}")


    return results


def print_results_table(sql_base, sql_fix, bash_base, bash_fix):
    print("\n" + "="*85)
    print(f"{'Method':<18} {'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7}"
          f" | {'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7}")
    print(f"{'':18} {'--- SQL ---':^33} | {'--- Bash ---':^33}")
    print("="*85)
    for label, sq, ba in [("Base",        sql_base, bash_base),
                           ("SLMFix-GRPO", sql_fix,  bash_fix)]:
        print(
            f"{label:<18}"
            f" {sq['bleu']:>7.4f}"
            f" {sq['code_bert_score']:>7.4f}"
            f" {sq['pass_rate']*100:>6.1f}%"
            f" {sq['ast_diff']:>7.4f}"
            f" | {ba['bleu']:>7.4f}"
            f" {ba['code_bert_score']:>7.4f}"
            f" {ba['pass_rate']*100:>6.1f}%"
            f" {ba['ast_diff']:>7.4f}"
        )
    print("="*85)


print("Evaluation functions ready.")

Evaluation functions ready.


**Cell 29 -- Smoke Test (10 samples)**

In [ ]:
# Force proposer back to GPU
proposer_model.to("cuda")
eval_fixer.to("cpu")
torch.cuda.empty_cache()
gc.collect()
print("proposer_model device:", next(proposer_model.parameters()).device)
print("eval_fixer device:", next(eval_fixer.parameters()).device)

proposer_model device: cuda:0
eval_fixer device: cpu


In [ ]:
print("="*50)
print("SMOKE TEST — 10 samples each")
print("="*50)

sql_base_test  = evaluate_slmfix(dev_sdata, "sql",  max_samples=10, use_fixer=False)
sql_fix_test   = evaluate_slmfix(dev_sdata, "sql",  max_samples=10, use_fixer=True)
bash_base_test = evaluate_slmfix(dev_bdata, "bash", max_samples=10, use_fixer=False)
bash_fix_test  = evaluate_slmfix(dev_bdata, "bash", max_samples=10, use_fixer=True)

def print_results_table(sql_base, sql_fix, bash_base, bash_fix):
    print("\n" + "="*85)
    print(f"{'Method':<18} {'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7}"
          f" | {'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7}")
    print(f"{'':18} {'--- SQL ---':^33} | {'--- Bash ---':^33}")
    print("="*85)
    for label, sq, ba in [("Base",        sql_base, bash_base),
                           ("SLMFix-GRPO", sql_fix,  bash_fix)]:
        print(
            f"{label:<18}"
            f" {sq['bleu']:>7.4f}"
            f" {sq['code_bert_score']:>7.4f}"
            f" {sq['pass_rate']*100:>6.1f}%"
            f" {sq['ast_diff']:>7.4f}"
            f" | {ba['bleu']:>7.4f}"
            f" {ba['code_bert_score']:>7.4f}"
            f" {ba['pass_rate']*100:>6.1f}%"
            f" {ba['ast_diff']:>7.4f}"
        )
    print("="*85)



print_results_table(sql_base_test, sql_fix_test, bash_base_test, bash_fix_test)
print("\n✅ Smoke test done — run full eval in next cell")

SMOKE TEST — 10 samples each

Evaluating SQL | Samples: 10 | Mode: Base


100%|██████████| 10/10 [00:55<00:00,  5.54s/it]

Computing CodeBERTScore...


config.json:   0%|          | 0.00/703 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...


Results — BASH | Base
  Pass Rate:      90.00%
  BLEU:           0.1511
  CodeBERTScore:  0.8661
  AST Similarity: 0.5849

Evaluating BASH | Samples: 10 | Mode: SLMFix-GRPO


100%|██████████| 10/10 [00:29<00:00,  2.96s/it]


Computing CodeBERTScore...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Results — BASH | SLMFix-GRPO
  Pass Rate:      100.00%
  BLEU:           0.1384
  CodeBERTScore:  0.8412
  AST Similarity: 0.5538

Method                BLEU     CBS   Pass%     AST |    BLEU     CBS   Pass%     AST
                              --- SQL ---            |           --- Bash ---           
Base                0.1341  0.8924  100.0%  0.3029 |  0.1511  0.8661   90.0%  0.5849
SLMFix-GRPO         0.1341  0.8924  100.0%  0.3029 |  0.1384  0.8412  100.0%  0.5538

✅ Smoke test done — run full eval in next cell


**Cell 30 -- Full Eval (SLMFIX-GRPO + Base)**

In [ ]:
#sql base
proposer_model.to("cuda")
eval_fixer.to("cpu")
torch.cuda.empty_cache()
gc.collect()

sql_base = evaluate_slmfix(dev_sdata, "sql", max_samples=300, use_fixer=False)
with open(os.path.join(PROJECT_ROOT, "generated_data/sql_base.json"), "w") as f:
    json.dump(sql_base, f, indent=2)
print("✅ sql_base saved →", sql_base)


Evaluating SQL | Samples: 300 | Mode: Base


100%|██████████| 300/300 [19:36<00:00,  3.92s/it]


Computing CodeBERTScore...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Results — SQL | Base
  Pass Rate:      88.67%
  BLEU:           0.0811
  CodeBERTScore:  0.8438
  AST Similarity: 0.1242
✅ sql_base saved → {'pass_rate': 0.8866666666666667, 'bleu': 0.08106361703956307, 'code_bert_score': 0.8438202738761902, 'ast_diff': 0.12422373998337259}


In [ ]:
#sql fix
proposer_model.to("cuda")
eval_fixer.to("cpu")
torch.cuda.empty_cache()
gc.collect()

sql_fix = evaluate_slmfix(dev_sdata, "sql", max_samples=300, use_fixer=True)
with open(os.path.join(PROJECT_ROOT, "generated_data/sql_fix.json"), "w") as f:
    json.dump(sql_fix, f, indent=2)
print("✅ sql_fix saved →", sql_fix)


Evaluating SQL | Samples: 300 | Mode: SLMFix-GRPO


100%|██████████| 300/300 [1:44:11<00:00, 20.84s/it]


Computing CodeBERTScore...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Results — SQL | SLMFix-GRPO
  Pass Rate:      88.67%
  BLEU:           0.0828
  CodeBERTScore:  0.8437
  AST Similarity: 0.1302
✅ sql_fix saved → {'pass_rate': 0.8866666666666667, 'bleu': 0.0827669804832183, 'code_bert_score': 0.843655526638031, 'ast_diff': 0.13015455041761229}


In [ ]:
#bash base
proposer_model.to("cuda")
eval_fixer.to("cpu")
torch.cuda.empty_cache()
gc.collect()

bash_base = evaluate_slmfix(dev_bdata, "bash", max_samples=300, use_fixer=False)
with open(os.path.join(PROJECT_ROOT, "generated_data/bash_base.json"), "w") as f:
    json.dump(bash_base, f, indent=2)
print("✅ bash_base saved →", bash_base)


Evaluating BASH | Samples: 300 | Mode: Base


100%|██████████| 300/300 [13:16<00:00,  2.65s/it]


Computing CodeBERTScore...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Results — BASH | Base
  Pass Rate:      98.00%
  BLEU:           0.1428
  CodeBERTScore:  0.8463
  AST Similarity: 0.5065
  Fix attempted:  0 / 300
  Fix recovered:  0 / 1
✅ bash_base saved → {'pass_rate': 0.98, 'bleu': 0.14279258159006555, 'code_bert_score': 0.8463225960731506, 'ast_diff': 0.5065336968883708, 'fix_attempted': 0, 'fix_recovered': 0}


In [ ]:
#bash fix
proposer_model.to("cuda")
eval_fixer.to("cpu")
torch.cuda.empty_cache()
gc.collect()

bash_fix = evaluate_slmfix(dev_bdata, "bash", max_samples=300, use_fixer=True)
with open(os.path.join(PROJECT_ROOT, "generated_data/bash_fix.json"), "w") as f:
    json.dump(bash_fix, f, indent=2)
print("✅ bash_fix saved →", bash_fix)


Evaluating BASH | Samples: 300 | Mode: SLMFix-GRPO


100%|██████████| 300/300 [13:57<00:00,  2.79s/it]


Computing CodeBERTScore...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Results — BASH | SLMFix-GRPO
  Pass Rate:      98.33%
  BLEU:           0.1424
  CodeBERTScore:  0.8455
  AST Similarity: 0.5055
  Fix attempted:  6 / 300
  Fix recovered:  1 / 6
✅ bash_fix saved → {'pass_rate': 0.9833333333333333, 'bleu': 0.14236800773745362, 'code_bert_score': 0.8454931974411011, 'ast_diff': 0.5054966598513339, 'fix_attempted': 6, 'fix_recovered': 1}


In [ ]:
#complete results output

import json

# Load from Drive in case of reconnect
def load_result(name):
    path = os.path.join(PROJECT_ROOT, f"generated_data/{name}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

sql_base  = load_result("sql_base")  or sql_base
sql_fix   = load_result("sql_fix")   or sql_fix
bash_base = load_result("bash_base") or bash_base
bash_fix  = load_result("bash_fix")  or bash_fix

print_results_table(sql_base, sql_fix, bash_base, bash_fix)

# Save combined
with open(os.path.join(PROJECT_ROOT, "generated_data/eval_results.json"), "w") as f:
    json.dump({
        "sql_base": sql_base, "sql_fix": sql_fix,
        "bash_base": bash_base, "bash_fix": bash_fix
    }, f, indent=2)
print("All results saved to Drive.")


Method                BLEU     CBS   Pass%     AST |    BLEU     CBS   Pass%     AST
                              --- SQL ---            |           --- Bash ---           
Base                0.0811  0.8438   88.7%  0.1242 |  0.1428  0.8463   98.0%  0.5065
SLMFix-GRPO         0.0828  0.8437   88.7%  0.1302 |  0.1424  0.8455   98.3%  0.5055
All results saved to Drive.


**Cell 31 -- Self-Correction Baseline**

In [ ]:
def self_correction_sql(question, schema, db_id):
    base_pred = generate_sql(question, schema, tok=tokenizer, mdl=proposer_model)
    val = validate_sql_combined(base_pred, db_id)
    if not val["fully_valid"]:
        msgs = [
            {"role": "system", "content": "Fix SQL. Output only SQL."},
            {"role": "user", "content":
                f"{question}\n{schema}\n{base_pred}\n{val.get('static_error','')}\n```sql"}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            out = proposer_model.generate(**inputs, max_new_tokens=150, do_sample=False,
                                          pad_token_id=tokenizer.eos_token_id)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return tokenizer.decode(gen[0], skip_special_tokens=True).strip()
    return base_pred

def self_correction_bash(nl):
    base_pred = generate_bash(nl, tok=tokenizer, mdl=proposer_model)
    val = validate_bash_static(base_pred)
    static_ok  = val["static_valid"] if isinstance(val, dict) else val[0]
    static_err = val.get("static_error") if isinstance(val, dict) else val[1]
    if not static_ok:
        msgs = [
            {"role": "system", "content": "Fix Bash. Output only command."},
            {"role": "user", "content":
                f"{nl}\n{base_pred}\n{static_err or ''}\n```bash"}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            out = proposer_model.generate(**inputs, max_new_tokens=100, do_sample=False,
                                          pad_token_id=tokenizer.eos_token_id)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return tokenizer.decode(gen[0], skip_special_tokens=True).strip()
    return base_pred

def evaluate_self_correction(dev_data, language, max_samples=300):
    dev_data = dev_data[:max_samples]
    pass_count, bleu_scores, ast_scores = 0, [], []
    predictions, gold_refs = [], []
    print(f"\nSelf-Correction | {language.upper()} | {len(dev_data)} samples")
    for sample in tqdm(dev_data):
        if language == "sql":
            db_id  = sample["db_id"]
            schema = get_db_schema(db_id)
            gold   = sample["query"]
            pred   = self_correction_sql(sample["question"], schema, db_id)
            if validate_sql_combined(pred, db_id)["fully_valid"]:
                pass_count += 1
            ast_scores.append(compute_sql_ast_similarity(pred, gold))
        else:
            gold = sample["bash"]
            pred = self_correction_bash(sample["nl"])
            v    = validate_bash_static(pred)
            if (v["static_valid"] if isinstance(v, dict) else v[0]):
                pass_count += 1
            ast_scores.append(compute_bash_ast_similarity(pred, gold))
        bleu_scores.append(compute_bleu(pred, gold))
        predictions.append(pred)
        gold_refs.append(gold)
    try:
        cbs_output = cbs_score(predictions, gold_refs, lang="python", verbose=False)
        _, _, cbs, _ = cbs_output if len(cbs_output) == 4 else (*cbs_output, None)
        cbs_mean = cbs.mean().item()
    except:
        cbs_mean = 0.0
    n = len(dev_data)
    results = {"pass_rate": pass_count/n, "bleu": sum(bleu_scores)/n,
               "code_bert_score": cbs_mean, "ast_diff": sum(ast_scores)/n}
    print(f"Pass%: {results['pass_rate']*100:.2f}% | BLEU: {results['bleu']:.4f} | "
          f"CBS: {results['code_bert_score']:.4f} | AST: {results['ast_diff']:.4f}")
    return results

proposer_model.to("cuda")
sc_sql_full  = evaluate_self_correction(dev_sdata, "sql")
with open(os.path.join(PROJECT_ROOT, "generated_data/sc_sql.json"), "w") as f:
    json.dump(sc_sql_full, f, indent=2)
print("✅ sc_sql saved")

sc_bash_full = evaluate_self_correction(dev_bdata, "bash")
with open(os.path.join(PROJECT_ROOT, "generated_data/sc_bash.json"), "w") as f:
    json.dump(sc_bash_full, f, indent=2)
print("✅ sc_bash saved")


Self-Correction | SQL | 300 samples


100%|██████████| 300/300 [20:29<00:00,  4.10s/it]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Pass%: 90.00% | BLEU: 0.0814 | CBS: 0.8436 | AST: 0.1242
✅ sc_sql saved

Self-Correction | BASH | 300 samples


100%|██████████| 300/300 [10:59<00:00,  2.20s/it]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Pass%: 98.33% | BLEU: 0.1428 | CBS: 0.8462 | AST: 0.5076
✅ sc_bash saved


**Cell 32 -- ICL Baseline**

In [ ]:
# ICL — 2 examples prepended to prompt, paper §5.3
SQL_ICL_EXAMPLES = [
    {"question": "How many singers are there?",
     "schema": "Table: singer, columns: singer_id, name, age",
     "gold": "SELECT count(*) FROM singer"},
    {"question": "List all singer names.",
     "schema": "Table: singer, columns: singer_id, name, age",
     "gold": "SELECT name FROM singer"},
]
BASH_ICL_EXAMPLES = [
    {"nl": "list all files in current directory", "gold": "ls"},
    {"nl": "print hello world",                  "gold": "echo 'hello world'"},
]

def icl_sql(question, schema):
    examples_text = ""
    for ex in SQL_ICL_EXAMPLES:
        examples_text += f"Question: {ex['question']}\nSchema: {ex['schema']}\nSQL: {ex['gold']}\n\n"
    msgs = [
        {"role": "system", "content": "You are an expert in SQL. Output only SQL."},
        {"role": "user", "content":
            f"{examples_text}Question: {question}\nSchema: {schema}\nSQL:"}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = proposer_model.generate(**inputs, max_new_tokens=150, do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen[0], skip_special_tokens=True).strip()

def icl_bash(nl):
    examples_text = ""
    for ex in BASH_ICL_EXAMPLES:
        examples_text += f"Task: {ex['nl']}\nBash: {ex['gold']}\n\n"
    msgs = [
        {"role": "system", "content": "You are an expert in Bash. Output only the command."},
        {"role": "user", "content": f"{examples_text}Task: {nl}\nBash:"}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = proposer_model.generate(**inputs, max_new_tokens=100, do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen[0], skip_special_tokens=True).strip()

def evaluate_icl(dev_data, language, max_samples=300):
    dev_data = dev_data[:max_samples]
    pass_count, bleu_scores, ast_scores = 0, [], []
    predictions, gold_refs = [], []
    print(f"\nICL | {language.upper()} | {len(dev_data)} samples")
    for sample in tqdm(dev_data):
        if language == "sql":
            db_id  = sample["db_id"]
            schema = get_db_schema(db_id)
            gold   = sample["query"]
            pred   = icl_sql(sample["question"], schema)
            if validate_sql_combined(pred, db_id)["static_valid"]:
                pass_count += 1
            ast_scores.append(compute_sql_ast_similarity(pred, gold))
        else:
            gold = sample["bash"]
            pred = icl_bash(sample["nl"])
            v    = validate_bash_static(pred)
            if (v["static_valid"] if isinstance(v, dict) else v[0]):
                pass_count += 1
            ast_scores.append(compute_bash_ast_similarity(pred, gold))
        bleu_scores.append(compute_bleu(pred, gold))
        predictions.append(pred)
        gold_refs.append(gold)
    try:
        cbs_output = cbs_score(predictions, gold_refs, lang="python", verbose=False)
        _, _, cbs, _ = cbs_output if len(cbs_output) == 4 else (*cbs_output, None)
        cbs_mean = cbs.mean().item()
    except:
        cbs_mean = 0.0
    n = len(dev_data)
    results = {"pass_rate": pass_count/n, "bleu": sum(bleu_scores)/n,
               "code_bert_score": cbs_mean, "ast_diff": sum(ast_scores)/n}
    print(f"Pass%: {results['pass_rate']*100:.2f}% | BLEU: {results['bleu']:.4f} | "
          f"CBS: {results['code_bert_score']:.4f} | AST: {results['ast_diff']:.4f}")
    return results

proposer_model.to("cuda")
icl_sql_full  = evaluate_icl(dev_sdata, "sql")
with open(os.path.join(PROJECT_ROOT, "generated_data/icl_sql.json"), "w") as f:
    json.dump(icl_sql_full, f, indent=2)
print("✅ icl_sql saved")

icl_bash_full = evaluate_icl(dev_bdata, "bash")
with open(os.path.join(PROJECT_ROOT, "generated_data/icl_bash.json"), "w") as f:
    json.dump(icl_bash_full, f, indent=2)
print("✅ icl_bash saved")


ICL | SQL | 300 samples


100%|██████████| 300/300 [18:03<00:00,  3.61s/it]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Pass%: 99.67% | BLEU: 0.0915 | CBS: 0.8439 | AST: 0.1244
✅ icl_sql saved

ICL | BASH | 300 samples


100%|██████████| 300/300 [09:06<00:00,  1.82s/it]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Pass%: 98.00% | BLEU: 0.1908 | CBS: 0.8624 | AST: 0.4766
✅ icl_bash saved


**Cell 33 -- Final Results Table**

In [ ]:
import json

# Load all results from Drive in case of reconnect
def load_result(name):
    path = os.path.join(PROJECT_ROOT, f"generated_data/{name}.json")
    with open(path) as f:
        return json.load(f)

sql_base  = load_result("sql_base")
sql_fix   = load_result("sql_fix")
bash_base = load_result("bash_base")
bash_fix  = load_result("bash_fix")
sc_sql    = load_result("sc_sql")
sc_bash   = load_result("sc_bash")
icl_sql   = load_result("icl_sql")
icl_bash  = load_result("icl_bash")

def print_final_table(results_dict):
    print("\n" + "="*85)
    print(f"{'':22} {'SQL':^38} {'Bash':^38}")
    print(f"{'Method':22} "
          f"{'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7} | "
          f"{'BLEU':>7} {'CBS':>7} {'Pass%':>7} {'AST':>7}")
    print("="*85)
    for name, res in results_dict.items():
        sr, br = res["sql"], res["bash"]
        print(f"{name:22} "
              f"{sr['bleu']:.4f} {sr['code_bert_score']:.4f} "
              f"{sr['pass_rate']*100:6.2f}% {sr['ast_diff']:.4f} | "
              f"{br['bleu']:.4f} {br['code_bert_score']:.4f} "
              f"{br['pass_rate']*100:6.2f}% {br['ast_diff']:.4f}")
    print("="*85)

all_results = {
    "Base":            {"sql": sql_base,  "bash": bash_base},
    "Self-Correction": {"sql": sc_sql,    "bash": sc_bash},
    "ICL":             {"sql": icl_sql,   "bash": icl_bash},
    "SLMFix-GRPO":     {"sql": sql_fix,   "bash": bash_fix},
}
print_final_table(all_results)

# Save final combined results
with open(os.path.join(PROJECT_ROOT, "generated_data/all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print("\n✅ All results saved to Drive → generated_data/all_results.json")
print("🎉 Project complete.")


                                        SQL                                    Bash                 
Method                    BLEU     CBS   Pass%     AST |    BLEU     CBS   Pass%     AST
Base                   0.0811 0.8438  88.67% 0.1242 | 0.1428 0.8463  98.00% 0.5065
Self-Correction        0.0814 0.8436  90.00% 0.1242 | 0.1428 0.8462  98.33% 0.5076
ICL                    0.0915 0.8439  99.67% 0.1244 | 0.1908 0.8624  98.00% 0.4766
SLMFix-GRPO            0.0828 0.8437  88.67% 0.1302 | 0.1424 0.8455  98.33% 0.5055

✅ All results saved to Drive → generated_data/all_results.json
🎉 Project complete.
